# Exercise 5 - LLMs for Content Analysis
_By Abigail Hayes_

In this exercise we will look at content analysis and labelling using LLMs. We will also examine handling validation and inter-annotator agreement.

We will use the same LLM as in the last 2 weeks. **You will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of two parts:
1. LLM Annotation
2. Performance analysis

## Setup

Follow the instructions from Exercise 3 for LLM setup. Make sure you are using the correct kernel is selected for this notebook.

You will need to install kagglehub in the terminal for downloading the dataset, or download it manually.

``source llm4ess_env/bin/activate``
``pip install kagglehub``

### Human annotation

Before we begin, please complete the following [survey](https://forms.gle/b75KKFM6vD2pcWER8). We will use this later for evaluating the model's responses.

## LLM Annotation

We are now familar with getting the LLM to provide us with responses to questions through prompting. The dataset we are using is from Kaggle and provides us with a [collection of news headlines](https://www.kaggle.com/datasets/rmisra/news-category-dataset).

First we download the data.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-category-dataset")

print("Path to dataset files:", path)

And then we get the same selection you have seen in the survey.

In [ ]:
import pandas as pd
import os

file_path = os.path.join(path, "News_Category_Dataset_v3.json")
df = pd.read_json(file_path, lines=True)  
df = df[df['category']=='WORLD NEWS']

sample_df = df.sample(n=30, random_state=7) 

for hl in sample_df['headline']:
    print(hl)

In [ ]:
system_prompt = "You are a sentiment classifier. Answer only with the label as a single word. The sentiment can be positive, negative or neutral."
user_question = ". What is the sentiment?"

user_prompts = []
for hl in sample_df['headline']:
    user_prompts.append(hl + user_question)

batch_inputs = [
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    for user_prompt in user_prompts
]

print(batch_inputs[0])

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="allenai/OLMo-2-0425-1B-Instruct")

In [ ]:
from transformers import set_seed

for seed in range(1,4):
    set_seed(seed)
    
    outputs = pipe(batch_inputs, max_new_tokens=100)
    
    llm_label = []
    for conversation in outputs:
        reply = conversation[0]['generated_text'][-1]['content']

        if reply.lower() in ['positive','negative','neutral']:
            llm_label.append(reply.lower())
        else:
            llm_label.append('error')
            print(reply)
    sample_df['llm_'+str(seed)] = llm_label

In [ ]:
sample_df

## Performance analysis

First let us look at how many errors we got.

In [ ]:
error_count = (sample_df[['llm_1', 'llm_2', 'llm_3']] == 'error').sum().sum()
print("Total 'error' count:", error_count)

How often does the LLM always get the same label? Or all 3 different?

In [ ]:
same_count = (sample_df['llm_1'] == sample_df['llm_2']) & (sample_df['llm_2'] == sample_df['llm_3'])
same_count = same_count.sum()

print("Number of rows where all 3 LLM outputs are identical:", same_count)

In [ ]:
all_different = (
    (sample_df['llm_1'] != sample_df['llm_2']) &
    (sample_df['llm_1'] != sample_df['llm_3']) &
    (sample_df['llm_2'] != sample_df['llm_3'])
)

count_all_different = all_different.sum()
print("Number of rows where all 3 LLM outputs differ:", count_all_different)


### Human annotator agreement

Now we would like to compare to our ground-truth. We will create this from the human annotated data, but we need to first look at how good this is...

In [ ]:
human_data = pd.read_csv('News headline sentiment (Responses) - Form responses 1.csv')
selected_cols = human_data.columns[1:-2]
df_selected = human_data[selected_cols]
df_selected = df_selected.astype(str).apply(lambda col: col.str.lower())

# Transpose: columns become rows, rows become columns
human_df = pd.DataFrame({
    "headline": selected_cols
})

# Add each row from df_selected as a new column
for i, row in enumerate(df_selected.itertuples(index=False), 1):
    human_df[f"annotator_{i}"] = row

human_df

Now to look at agreement with Krippendorff's alpha.

In [ ]:
import numpy as np

annotator_cols = [col for col in human_df.columns if col.startswith("annotator_")]
data = human_df[annotator_cols]

def k_alpha(data):
    # Get all unique categories
    categories = pd.unique(data.values.ravel())
    
    # Create one-hot encoding for categories
    one_hot = pd.get_dummies(data)
    
    # Convert to numpy
    one_hot = one_hot.to_numpy()
    
    n_items, n_annotators = data.shape
    
    # Total pairwise comparisons per item
    pairwise_counts = n_annotators * (n_annotators - 1)
    
    D_o = 0
    for row in data.to_numpy():
        # Count how many pairs disagree
        for i in range(n_annotators):
            for j in range(i+1, n_annotators):
                if row[i] != row[j]:
                    D_o += 1
    
    # Normalize by total pairs
    D_o /= (n_items * pairwise_counts)
    
    # Flatten all annotations
    all_labels = data.values.ravel()
    freqs = pd.Series(all_labels).value_counts(normalize=True)
    
    # Expected disagreement = 1 - sum of squared frequencies
    D_e = 1 - np.sum(freqs**2)
    
    alpha = 1 - D_o / D_e
    print("Krippendorff's alpha:", alpha)

k_alpha(data)

How many does everyone agree on? How does this compare to your guesses?

In [ ]:
# Check if all values in the row are the same
full_agreement = human_df[annotator_cols].nunique(axis=1) == 1

# Count rows with full agreement
num_full_agreement = full_agreement.sum()

print("Number of rows with full agreement:", num_full_agreement)

human_data.iloc[:, -1].value_counts().plot.bar(figsize=(8,5), color='skyblue', rot=45)

### Task

Try out a method for selecting the human 'correct' label. What is the model performance?

What about if you only keep questions with high human performance? Or give some credit for matching with any human?